In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)

import hashlib
import pickle
from pathlib import Path

from sklearn.neural_network import MLPRegressor

from model import hybrid_system_exp, grid_search_exp

import config
%load_ext autoreload
%autoreload 2

In [ ]:
# === Notebook ONE-OFF -- baseline hibrido ARIMA-MLP (Additive) SO para as 2 ===
# series novas (windspeedfortaleza, samurec), adicionadas em 2026-08-28.
#
# NAO e o notebook de baseline protegido de 17 series -- nao importa, nao
# referencia, nao compartilha lista com ele (RUNBOOK.md Secao 6 / CLAUDE.md
# Secao 3). Replica a MESMA logica (mesmos hiperparametros, model_exec=10,
# mesma chamada de GridSearch com hybrid_system_exp.Additive), restrita a 2
# series e com force=False.
model = MLPRegressor(activation='logistic', solver='lbfgs')

experiment_id = 'chamados'
model_name = 'amv1'
normalize = True
force = False          # OBRIGATORIO: nunca True aqui.
model_exec = 10

experiment_params = {
    'linear_model_name': '1arima',
    'diff_kpss': False,
    'horizon': 1,
}

model_parameters = {
    'hidden_layer_sizes': [10, 20, 50],
    'max_iter': [1000],
}

new_series_list = ['windspeedfortaleza.txt', 'samurec.txt']

chamados_dir = Path(config.ROOT_PATH) / 'data' / 'result' / 'chamados'

# --- Guarda pre-flight: Additive precisa do ARIMA pre-treinado sob 'chamados' ---
# (experiment_params['linear_model_name'] == '1arima'). Se este notebook rodar
# ANTES de arima_exec.ipynb ter gerado o ARIMA das 2 series, falha aqui com
# mensagem clara em vez de um FileNotFoundError cru la dentro do Additive.
missing_arima = [
    s for s in new_series_list
    if not (chamados_dir / f"{s.replace('.txt', '')}_1arima.pkl").exists()
]
assert not missing_arima, (
    f"Falta o ARIMA pre-treinado para {missing_arima} em data/result/chamados/. "
    "Rode arima_exec.ipynb PRIMEIRO (as 2 series ja estao em config.BASE_NAME_LIST; "
    "com force=False as demais sao puladas)."
)

# --- Snapshot de hash ANTES: TODOS os .pkl de chamados/ ---
hashes_before = {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in sorted(chamados_dir.glob('*.pkl'))
}
print(f"snapshot: {len(hashes_before)} .pkl em chamados/ antes da execucao")

for base_name in new_series_list:
    print(base_name)
    exec_gs = grid_search_exp.GridSearch(
        hybrid_system_exp.Additive,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
    )
    exec_gs.execution()

In [ ]:
# === VERIFICACAO POS-EXECUCAO (versao 2 series, snapshot in-notebook) ===
# Adaptada da celula de verificacao dos notebooks de baseline de 17 series
# (que espera 17 series + um arquivo de snapshot de hash especifico -- nao se
# aplica aqui). Garantia essencial preservada: confirma que NENHUM baseline
# pre-existente foi tocado.
assert len(hashes_before) > 0, (
    "hashes_before vazio -- a celula de configuracao nao rodou, ou chamados/ "
    "estava vazio. Sem baselines pre-existentes para comparar, a checagem (b) "
    "passaria trivialmente sem valor. Rode a celula anterior primeiro."
)

expected_new = {'samurec_1amv1.pkl', 'windspeedfortaleza_1amv1.pkl'}

hashes_after = {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in sorted(chamados_dir.glob('*.pkl'))
}

created = set(hashes_after) - set(hashes_before)
touched_existing = {n for n in hashes_before if hashes_after.get(n) != hashes_before[n]}

print("--- (a) arquivos criados ---")
print(f"  criados: {sorted(created)}")
assert created == expected_new, (
    f"esperava criar exatamente {sorted(expected_new)}, criou {sorted(created)}"
)

print("--- (b) nenhum .pkl pre-existente foi tocado (TODOS os baselines, nao so MLP) ---")
print(f"  pre-existentes alterados: {sorted(touched_existing)}")
assert not touched_existing, (
    f"ALERTA: {sorted(touched_existing)} mudaram de hash -- este one-off (force=False) "
    "NAO deveria tocar nenhum baseline pre-existente. Investigar antes de prosseguir."
)

print("--- (c) config persistida nos 2 .pkl novos ---")
for name in sorted(expected_new):
    with open(chamados_dir / name, 'rb') as f:
        saved = pickle.load(f)
    obj = saved[0]['experiment']
    act = obj.model.activation
    print(f"  {name}: activation={act!r}")
    assert act == 'logistic', f"{name}: esperado 'logistic', achou {act!r}"

print()
print("OK -- 2 baselines novos criados; 0 baseline protegido tocado.")